In [6]:
# First, install the required package
!pip install zstandard

import io
import zstandard as zstd  # Using the correct import for zstandard
import json
import os
import pandas as pd
import glob

def filter_dump(input_file, output_folder):
    """Process a Reddit dump file and extract relevant posts"""
    print(f"\nProcessing {input_file}...")
    
    # Initialize results dictionary to store posts by comeback name
    results = {}
    
    # Open the zst compressed file
    with open(input_file, 'rb') as file_obj:
        # Create a zstandard decompression context
        dctx = zstd.ZstdDecompressor()
        # Create a stream reader
        with dctx.stream_reader(file_obj) as reader:
            # Wrap the reader with TextIOWrapper for line-by-line reading
            with io.TextIOWrapper(reader, encoding='utf-8') as text_reader:
                for i, line in enumerate(text_reader):
                    if i % 100000 == 0:
                        print(f"  Processed {i:,} lines")
                        # Write intermediate results to disk to save memory
                        write_results_to_disk(results, output_folder)
                    
                    try:
                        # Parse the JSON line
                        data = json.loads(line)
                        
                        # Your filtering logic here
                        # This is a placeholder - replace with your actual filtering logic
                        # For example, checking if the post contains certain keywords
                        
                        # Example: Add post to results if it contains a comeback name
                        for comeback_name in ["example_comeback1", "example_comeback2"]:
                            if comeback_name in data.get("body", "").lower() or comeback_name in data.get("title", "").lower():
                                if comeback_name not in results:
                                    results[comeback_name] = []
                                results[comeback_name].append(data)
                        
                        # Periodically write to disk
                        for comeback_name, posts in list(results.items()):
                            if len(posts) >= 5000:  # Adjust threshold as needed
                                write_specific_results_to_disk(comeback_name, posts, output_folder)
                                results[comeback_name] = []  # Clear after writing
                                
                    except json.JSONDecodeError:
                        continue
    
    # Write any remaining results to disk
    write_results_to_disk(results, output_folder)
    print(f"Finished processing {input_file}")

def write_specific_results_to_disk(comeback_name, posts, output_folder):
    """Write results for a specific comeback to disk"""
    if posts:
        df = pd.DataFrame(posts)
        out_path = os.path.join(output_folder, f"{comeback_name}_reddit_raw.csv")
        if os.path.exists(out_path):
            # Append if file already exists from previous dump
            df.to_csv(out_path, mode="a", header=False, index=False)
        else:
            df.to_csv(out_path, index=False)
        print(f"  {comeback_name}: {len(posts)} posts saved")

def write_results_to_disk(results, output_folder):
    """Write the current results to disk and clear memory"""
    for comeback_name, posts in list(results.items()):
        if posts:
            write_specific_results_to_disk(comeback_name, posts, output_folder)

def is_file_processed(input_file, output_folder):
    """Check if a file has been fully processed by looking for corresponding output files"""
    # Get the base filename without extension
    base_name = os.path.basename(input_file).replace('.zst', '')
    
    # Check if there's a marker file indicating this file was processed
    marker_file = os.path.join(output_folder, f"{base_name}_processed.marker")
    if os.path.exists(marker_file):
        return True
    
    # Alternative: Check if output files exist for this input file
    # This depends on your specific naming convention
    # Example: Check if any files contain the input filename in their name
    output_files = glob.glob(os.path.join(output_folder, f"*{base_name}*"))
    return len(output_files) > 0

def mark_file_as_processed(input_file, output_folder):
    """Create a marker file to indicate that processing is complete"""
    base_name = os.path.basename(input_file).replace('.zst', '')
    marker_file = os.path.join(output_folder, f"{base_name}_processed.marker")
    with open(marker_file, 'w') as f:
        f.write(f"Processed on {pd.Timestamp.now()}")

# --- Run filter on comments and submissions ---
comments_folder = "../01_raw_data/reddit/comments"
submissions_folder = "../01_raw_data/reddit/submissions"
output_folder = "../01_raw_data/reddit"

# Create output folder if it doesn't exist
os.makedirs(output_folder, exist_ok=True)

# Define SKIP_FILES if not already defined
SKIP_FILES = []  # Add files to skip if needed

for folder, label in [(comments_folder, "comments"), (submissions_folder, "submissions")]:
    files = sorted([f for f in os.listdir(folder) if f.endswith(".zst")])
    print(f"\nFound {len(files)} {label} files: {files}")
    
    for f in files:
        input_file = os.path.join(folder, f)
        
        # Skip files that are in the SKIP_FILES list
        if f in SKIP_FILES:
            print(f"  Skipping {f} (in SKIP_FILES list)")
            continue
        
        # Skip files that have already been processed
        if is_file_processed(input_file, output_folder):
            print(f"  Skipping {f} (already processed)")
            continue
        
        # Process the file
        try:
            filter_dump(input_file, output_folder)
            # Mark the file as processed
            mark_file_as_processed(input_file, output_folder)
        except Exception as e:
            print(f"  Error processing {f}: {e}")

print("\nAll done!")


Found 5 comments files: ['RC_2024-07.zst', 'RC_2024-10.zst', 'RC_2024-11.zst', 'RC_2024-12.zst', 'RC_2025-02.zst']

Processing ../01_raw_data/reddit/comments\RC_2024-07.zst...
  Processed 0 lines
  Processed 100,000 lines
  Processed 200,000 lines
  Processed 300,000 lines
  Processed 400,000 lines
  Processed 500,000 lines
  Processed 600,000 lines
  Processed 700,000 lines
  Processed 800,000 lines
  Processed 900,000 lines
  Processed 1,000,000 lines
  Processed 1,100,000 lines
  Processed 1,200,000 lines
  Processed 1,300,000 lines
  Processed 1,400,000 lines
  Processed 1,500,000 lines
  Processed 1,600,000 lines
  Processed 1,700,000 lines
  Processed 1,800,000 lines
  Processed 1,900,000 lines
  Processed 2,000,000 lines
  Processed 2,100,000 lines
  Processed 2,200,000 lines
  Processed 2,300,000 lines
  Processed 2,400,000 lines
  Processed 2,500,000 lines
  Processed 2,600,000 lines
  Processed 2,700,000 lines
  Processed 2,800,000 lines
  Processed 2,900,000 lines
  Process

In [2]:
# ============================================================
# IVE Rebel Heart - Targeted Reddit Search
# February 2025 archive only
# ============================================================

import zstandard as zstd
import json
import pandas as pd
import os
import io
from datetime import datetime, timezone

IVE_COMEBACK = {
    "start": "2025-01-13",
    "end": "2025-02-10",  # Extended window to capture more data
    "keywords": [
        "ive", "rebel heart", "ive empathy", "wonyoung", "yujin",
        "leeseo", "gaeul", "rei ive", "liz ive", "starship entertainment",
        "ive comeback", "dive ive", "ive kpop", "ive group"
    ]
}

TARGET_SUBREDDITS = [
    "kpop", "kpopthoughts", "unpopularkpopopinions", "ive"
]

BOT_ACCOUNTS = [
    "AutoModerator", "automoderator", "RemindMeBot",
    "WikiTextBot", "converter-bot", "sneakpeek_bot"
]

def parse_date(date_str):
    return datetime.strptime(date_str, "%Y-%m-%d").replace(tzinfo=timezone.utc)

results = []
start_dt = parse_date(IVE_COMEBACK["start"])
end_dt = parse_date(IVE_COMEBACK["end"])

for filename, folder in [("RC_2025-02.zst", "../01_raw_data/reddit/comments"),
                          ("RS_2025-02.zst", "../01_raw_data/reddit/submissions")]:
    filepath = os.path.join(folder, filename)
    print(f"Processing {filename}...")
    count = 0

    with open(filepath, "rb") as f:
        dctx = zstd.ZstdDecompressor()
        with dctx.stream_reader(f) as reader:
            with io.TextIOWrapper(reader, encoding="utf-8") as text_reader:
                for line in text_reader:
                    try:
                        post = json.loads(line)

                        # Filter bots
                        author = post.get("author", "")
                        if author in BOT_ACCOUNTS or author.lower().endswith("bot"):
                            continue

                        # Filter deleted
                        if post.get("body", "") in ["[deleted]", "[removed]"] and \
                           post.get("selftext", "") in ["[deleted]", "[removed]", ""]:
                            continue

                        # Check date
                        created = datetime.fromtimestamp(
                            post.get("created_utc", 0), tz=timezone.utc
                        )
                        if not (start_dt <= created <= end_dt):
                            continue

                        # Check subreddit
                        subreddit = post.get("subreddit", "").lower()
                        if subreddit not in TARGET_SUBREDDITS:
                            continue

                        # Check keywords
                        text = (
                            post.get("title", "") + " " +
                            post.get("body", "") + " " +
                            post.get("selftext", "")
                        ).lower()

                        if any(kw in text for kw in IVE_COMEBACK["keywords"]):
                            results.append({
                                "id": post.get("id"),
                                "subreddit": subreddit,
                                "text": text.strip(),
                                "score": post.get("score", 0),
                                "created_utc": post.get("created_utc"),
                                "comeback": "IVE_RebelHeart"
                            })
                            count += 1

                    except json.JSONDecodeError:
                        continue

    print(f"  Found {count} IVE posts in {filename}")

print(f"\nTotal IVE posts: {len(results)}")

if results:
    df = pd.DataFrame(results)
    out_path = "../01_raw_data/reddit/IVE_RebelHeart_reddit_raw.csv"
    df.to_csv(out_path, index=False)
    print(f"Saved to {out_path}")
else:
    print("No IVE posts found - check keywords or date range")

Processing RC_2025-02.zst...
  Found 4228 IVE posts in RC_2025-02.zst
Processing RS_2025-02.zst...
  Found 548 IVE posts in RS_2025-02.zst

Total IVE posts: 4776
Saved to ../01_raw_data/reddit/IVE_RebelHeart_reddit_raw.csv
